# Plain TRL GRPO Experiment

This notebook is a clean experiment notebook, separate from `grpo_training.ipynb`.

Goal:
- start from plain `Qwen/Qwen3-4B`
- no 4-bit quantization
- add fresh LoRA
- do a short JSON-format SFT warm-up
- run a very small live poke-env rollout
- run GRPO on that real rollout data

This is the notebook to answer one question:
can GRPO actually run end-to-end on your task when we avoid the unstable quantized path?


In [ ]:
# Cell 1: repo / Colab setup
import os, sys, subprocess

GITHUB_REPO = "Atharva2099/Smogon-RL"
REPO_DIR = "/content/Smogon-RL"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", f"https://github.com/{GITHUB_REPO}.git", REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print(f"Repo ready at {REPO_DIR}")
else:
    REPO_DIR = os.path.abspath(".")
    print(f"Local run detected. Using repo at: {REPO_DIR}")


In [ ]:
# Cell 2: install pinned plain TRL stack
import sys

if IN_COLAB:
    %pip uninstall -y unsloth vllm torchaudio torchvision xformers || true
    %pip install -q "torch==2.10.0" "transformers==4.57.1" "trl==0.24.0" "peft==0.18.1" "accelerate>=1.0.0" "bitsandbytes"
    %pip install -q "poke-env==0.8.3" pydantic numpy datasets

    import subprocess as _sp
    _sp.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    print("Pinned plain TRL stack installed.")
else:
    import subprocess as _sp
    _sp.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    print("Local editable install complete.")


In [ ]:
# Cell 3: version check
import torch, transformers, trl, peft, bitsandbytes, accelerate
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("trl", trl.__version__)
print("peft", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)
print("accelerate", accelerate.__version__)


In [13]:
# Cell 4: experiment config
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
BASE_ADAPTER_REPO = "Atharva2099/openenv-smogon-rl"
BASE_ADAPTER_REVISION = "grpo-qwen3-4b-run2"
NEXT_ADAPTER_REVISION = "grpo-qwen3-4b-run3"
ADAPTER_DIR = "./openenv_grpo_lora_run3"
ROLLOUT_PATH = "./rollout_buffer_run3.jsonl"
MAX_SEQ_LENGTH = 1536
LOAD_IN_4BIT = False

LORA_RANK = 32
LORA_ALPHA = 32
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# More diversity so GRPO groups get real contrast.
NUM_GENERATIONS = 4
MAX_NEW_TOKENS = 24
TEMPERATURE = 0.3

LEARNING_RATE = 1e-5
LR_SCHEDULER_TYPE = "cosine"
OPTIM = "adamw_8bit"
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
MAX_STEPS = 64

# Warm-up needs to be strong enough that rollout is not 98% fallback.
JSON_WARMUP_STEPS = 160
RUN_BATTLES = 20

BATTLE_FORMAT = "gen4randombattle"

SYSTEM_PROMPT = (
    "You are a competitive Pokemon battler. "
    "Analyse the battle state below and choose exactly one action. "
    "You MUST output ONLY a single JSON object — no explanation, no markdown fences, just the raw JSON.\n"
    '{"action": "move" | "switch", "choice": "Exact Name of Move or Pokemon"}'
)


In [14]:
# Cell 5: load plain model + previous GRPO adapter
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

_token = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=_token,
    use_fast=True,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    token=_token,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    model,
    BASE_ADAPTER_REPO,
    revision=BASE_ADAPTER_REVISION,
    token=_token,
    is_trainable=True,
    autocast_adapter_dtype=False,
)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

for _, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float16 if torch.cuda.is_available() else torch.float32)

if hasattr(model, "lm_head"):
    try:
        model.lm_head.to(torch.float16 if torch.cuda.is_available() else torch.float32)
    except Exception:
        pass

model.train()
if hasattr(model, "config"):
    model.config.use_cache = False

print(
    f"Loaded adapter {BASE_ADAPTER_REPO}@{BASE_ADAPTER_REVISION} | "
    f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/132M [00:00<?, ?B/s]

Loaded adapter Atharva2099/openenv-smogon-rl@grpo-qwen3-4b-run2 | trainable params: 66,060,288


In [15]:
# Cell 6: repo imports + environment helpers
import os
import sys
import json
import re
import random
import subprocess
import time
import socket
import torch
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer

for _p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from smogon_rl.config import EnvConfig
from smogon_rl.openenv_sync_env import PokemonShowdownEnv
from smogon_rl.action_space import (
    ActionOption,
    build_action_instructions,
    enumerate_actions,
    extract_action_json_from_text,
    parse_llm_action,
)

_ACTION_RE = re.compile(
    r'\{\s*"action"\s*:\s*"(?:move|switch)"\s*,\s*"choice"\s*:\s*"[^"]+"\s*\}'
)

def _extract_content(completion) -> str:
    if isinstance(completion, list):
        text = completion[0].get("content", "")
    else:
        text = str(completion)
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def format_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        content = _extract_content(completion)
        rewards.append(0.1 if _ACTION_RE.search(content) else -2.0)
    return rewards

def game_reward(prompts, completions, collected_action, collected_reward, **kwargs):
    rewards = []
    for completion, ca, cr in zip(completions, collected_action, collected_reward):
        content = _extract_content(completion)
        try:
            parsed = json.loads(_ACTION_RE.search(content).group())
            ca_parsed = json.loads(ca)
            clipped = max(-0.5, min(2.0, float(cr)))
            if parsed == ca_parsed:
                rewards.append(1.0 + clipped)
            else:
                rewards.append(-0.25)
        except Exception:
            rewards.append(0.0)
    return rewards

def build_prompt_messages(state_str: str, valid_actions: list[ActionOption]) -> list[dict]:
    instructions = build_action_instructions(valid_actions)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{state_str}\n\n{instructions}"},
    ]

@torch.no_grad()
def generate_action_candidates(
    model,
    tokenizer,
    state_str: str,
    valid_actions: list[ActionOption],
    num_generations: int = NUM_GENERATIONS,
):
    messages = build_prompt_messages(state_str, valid_actions)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        chat_template_kwargs={"enable_thinking": False},
    )
    device = model.get_input_embeddings().weight.device
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_return_sequences=num_generations,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(out[input_len:], skip_special_tokens=True) for out in outputs]

print("Imports and helpers ready.")


Imports and helpers ready.


In [ ]:
# Cell 7: JSON warm-up SFT using rollout-shaped prompts
random.seed(42)

MOVES = [
    "thunderwave", "return", "thunderbolt", "fireblast", "icebeam", "earthquake",
    "surf", "recover", "calmmind", "uturn", "shadowball", "trick",
    "spore", "stealthrock", "rapidspin", "yawn", "toxic", "protect",
]
POKEMON = [
    "castform", "charizard", "arceusground", "parasect", "furret", "rotom",
    "kyogre", "torkoal", "mesprit", "registeel", "groudon", "metagross",
    "froslass", "slaking", "pikachu", "gyarados", "snorlax", "dragonite",
]
ITEMS = ["leftovers", "choiceband", "choicescarf", "lifeorb", "earthplate"]
ABILITIES = ["forecast", "blaze", "levitate", "pressure", "intimidate", "naturalcure"]
STATUS = ["Healthy", "Burned", "Poisoned", "Paralyzed"]


def _move_line(name):
    move_type = random.choice(list("NEWFGIBRPSD"))
    power = random.choice([0, 70, 75, 80, 95, 100, 120])
    return f"{name}({move_type}{power})"


def _party_block(active_name):
    roster = [active_name] + random.sample([p for p in POKEMON if p != active_name], 5)
    lines = []
    for idx, mon in enumerate(roster):
        hp = 100 if idx == 0 else random.choice([100, 88, 76, 64])
        item = random.choice(ITEMS)
        moves = " | ".join(_move_line(m) for m in random.sample(MOVES, 4))
        lines.append(f"- {mon} HP:{hp}% OK Item:{item}\n  Moves: {moves}")
    return lines, roster


def _build_realistic_example():
    active = random.choice(POKEMON)
    opponent = random.choice([p for p in POKEMON if p != active])
    self_lines, roster = _party_block(active)
    switch_opts = random.sample(roster[1:], k=random.randint(2, 4))
    move_opts = random.sample(MOVES, k=4)

    valid = [
        {"action": "move", "choice": m} for m in move_opts
    ] + [
        {"action": "switch", "choice": p} for p in switch_opts
    ]
    chosen = random.choice(valid)

    user_msg = (
        "## Part A: Active Field\n"
        f"### Active Self\n- Name: {active}\n- HP: {random.choice([100.0, 88.0, 74.0, 61.0])}%\n"
        f"- Status: {random.choice(STATUS)}\n- Ability: {random.choice(ABILITIES)}\n"
        f"- Item: {random.choice(ITEMS)}\n- Stat Modifiers: None\n"
        f"### Active Opponent\n- Name: {opponent}\n- HP: {random.choice([100.0, 79.0, 61.0, 44.0])}%\n"
        f"- Status: {random.choice(STATUS)}\n- Speed Range: {random.randint(120,220)}-{random.randint(240,360)}\n\n"
        "## Part B: Full Self Roster\n" + "\n".join(self_lines) + "\n\n"
        "## Part C: Opponent History\n"
        f"- {opponent} | Last HP: {random.choice([100.0, 79.0, 61.0])}% | Status: {random.choice(STATUS)}\n"
        f"  - Revealed item: unknown_item\n"
        f"  - Revealed ability: {random.choice(ABILITIES)}\n\n"
        "You must choose exactly one action and output pure JSON with this schema:\n\n"
        '{"action": "move" | "switch", "choice": "Exact Name of Move or Pokemon"}\n\n'
        "Valid options for this state:\n" + "\n".join(
            [f"- action: '{v['action']}', choice: '{v['choice']}'" for v in valid]
        )
    )

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": json.dumps(chosen)},
        ]
    }

examples = [_build_realistic_example() for _ in range(320)]
texts = [
    tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
    for ex in examples
]
sft_dataset = Dataset.from_dict({"text": texts})

sft_args = SFTConfig(
    output_dir="debug_plain_trl_sft",
    max_steps=JSON_WARMUP_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=10,
    report_to="none",
    packing=False,
    max_length=1024,
)

model.train()
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_args,
    train_dataset=sft_dataset,
)
trainer.train()
del trainer
print("JSON warm-up complete.")


In [16]:
import subprocess
import time
import os

SHOWDOWN_DIR = "/content/pokemon-showdown"

if not os.path.exists(SHOWDOWN_DIR):
    subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/smogon/pokemon-showdown.git", SHOWDOWN_DIR],
        check=True,
    )

subprocess.run(["pkill", "-f", "pokemon-showdown"], capture_output=True)
time.sleep(1)

showdown_proc = subprocess.Popen(
    ["node", "pokemon-showdown", "start", "--no-security"],
    cwd=SHOWDOWN_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

print(f"Pokémon Showdown server started (PID {showdown_proc.pid})")


2026-03-08 16:52:49,527 - RandomPlayer 14 - ERROR - no close frame received or sent
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/protocol.py", line 963, in transfer_data
    message = await self.read_message()
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/protocol.py", line 1033, in read_message
    frame = await self.read_data_frame(max_size=self.max_size)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/protocol.py", line 1108, in read_data_frame
    frame = await self.read_frame(max_size)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/protocol.py", line 1165, in read_frame
    frame = await Frame.read(
            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/framing.py", line 68, in read
  

Pokémon Showdown server started (PID 23821)


In [17]:
# Cell 8: start Showdown server + env

SHOWDOWN_DIR = "/content/pokemon-showdown"

if IN_COLAB and not os.path.exists(SHOWDOWN_DIR):
    subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/smogon/pokemon-showdown.git", SHOWDOWN_DIR],
        check=True,
    )

if IN_COLAB:
    subprocess.run(["pkill", "-f", "pokemon-showdown"], capture_output=True)
    time.sleep(1)
    showdown_proc = subprocess.Popen(
        ["node", "pokemon-showdown", "start", "--no-security"],
        cwd=SHOWDOWN_DIR,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(3)
    print(f"Showdown PID {showdown_proc.pid}")

_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
_server_up = _sock.connect_ex(("127.0.0.1", 8000)) == 0
_sock.close()
if not _server_up:
    raise RuntimeError("Showdown server not running on port 8000")

env_config = EnvConfig(
    battle_format=BATTLE_FORMAT,
    verbose_logging=True,
    log_every_n_steps=10,
    poll_heartbeat_seconds=5.0,
)
env = PokemonShowdownEnv(config=env_config)
print(env.config)


Showdown PID 23968
EnvConfig(battle_format='gen4randombattle', max_steps_per_battle=30, poll_interval_seconds=0.2, open_timeout=25.0, show_replays=False, verbose_logging=True, log_every_n_steps=10, poll_heartbeat_seconds=5.0, min_battle_reward=-100.0, max_no_progress_steps=2, step_living_penalty=-0.05, no_progress_termination_penalty=-1.0, max_steps_termination_penalty=-2.0)


In [18]:
# Cell 9: collect real rollout battles with checkpointing
def _load_rollout_buffer(path: str):
    if not os.path.exists(path):
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _append_rollout_row(path: str, row: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")

def collect_rollouts(
    model,
    tokenizer,
    env: PokemonShowdownEnv,
    num_battles: int = RUN_BATTLES,
    rollout_path: str = ROLLOUT_PATH,
):
    model.eval()
    buffer = _load_rollout_buffer(rollout_path)
    total_reward = 0.0
    total_turns = 0
    model_invalid_count = 0
    format_hits = 0

    if buffer:
        print(f"Loaded {len(buffer)} existing rollout rows from {rollout_path}", flush=True)

    for battle_idx in range(num_battles):
        print(f"[Rollout] Battle {battle_idx + 1}/{num_battles} started", flush=True)
        state_str = env.reset()
        done = False
        battle_turns = 0

        while not done:
            battle = env._ensure_battle()
            valid_actions = enumerate_actions(battle)
            if not valid_actions:
                print("No valid actions; ending battle early.", flush=True)
                break

            candidates = generate_action_candidates(model, tokenizer, state_str, valid_actions)
            chosen_str = None

            for c in candidates:
                clean = re.sub(r"<think>.*?</think>", "", c, flags=re.DOTALL).strip()
                try:
                    parse_llm_action(clean, valid_actions)
                    chosen_str = clean
                    format_hits += 1
                    break
                except ValueError:
                    pass

                extracted = extract_action_json_from_text(c)
                if extracted is not None:
                    try:
                        parse_llm_action(extracted, valid_actions)
                        chosen_str = extracted
                        format_hits += 1
                        break
                    except ValueError:
                        pass

            if chosen_str is None:
                model_invalid_count += 1
                fb = valid_actions[0]
                chosen_str = json.dumps({"action": fb.action_type, "choice": fb.choice})

            next_state_str, game_r, done, info = env.step(chosen_str)
            total_reward += game_r
            total_turns += 1
            battle_turns += 1

            row = {
                "prompt": build_prompt_messages(state_str, valid_actions),
                "collected_action": chosen_str,
                "collected_reward": float(game_r),
            }
            buffer.append(row)
            _append_rollout_row(rollout_path, row)
            state_str = next_state_str

            if battle_turns % 10 == 0 or done:
                print(
                    f"[Rollout] turn={battle_turns} "
                    f"game_turn={info.get('turn')} "
                    f"reward={game_r:.3f} "
                    f"env_illegal={info.get('action_illegal', False)}",
                    flush=True,
                )

        print(f"[Rollout] Battle finished in {battle_turns} turns", flush=True)

    print(
        f"Collected {len(buffer)} rows | "
        f"avg reward/turn={total_reward / max(1, total_turns):.3f} | "
        f"format hit rate={format_hits / max(1, total_turns) * 100:.1f}% | "
        f"model_invalid={model_invalid_count}"
    )
    return buffer

rollout_buffer = collect_rollouts(
    model,
    tokenizer,
    env,
    num_battles=RUN_BATTLES,
    rollout_path=ROLLOUT_PATH,
)
assert rollout_buffer, "No rollout rows collected"

dataset = Dataset.from_list(rollout_buffer)
print(dataset)
print(dataset[0])


[Rollout] Battle 1/20 started
[PokeEnvClient] Background event loop started.
[PokeEnvClient] Players created and connected.
[PokeEnvClient] Launching new battle in format gen4randombattle.
[PokeEnvClient] Battle update received: turn=1, finished=False.
[PokemonShowdownEnv] Battle 1 started at turn=1 (format=gen4randombattle).
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Waiting for turn advance: current_turn=1, previous_turn=1, elapsed=5.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=1, previous_turn=1, elapsed=10.0s.
[PokeEnvClient] Battle update received: turn=2, finished=False.
[PokemonShowdownEnv] battle=1 step=1 turn=2 reward=9.811 running_reward=9.811 illegal_action=False done=False
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Waiting for turn advance: current_turn=2, previous_turn=2, elapsed=5.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=2, previous_turn=2, elapsed=10.0s.
[PokeEnvClient] Waiting for turn adv

ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-3268' coro=<Player._send_challenges() running at /usr/local/lib/python3.12/dist-packages/poke_env/player/player.py:754> wait_for=<Future pending cb=[Task.task_wakeup()]> cb=[_chain_future.<locals>._call_set_state() at /usr/lib/python3.12/asyncio/futures.py:396]>
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-3269' coro=<Player._accept_challenges() running at /usr/local/lib/python3.12/dist-packages/poke_env/player/player.py:479> wait_for=<Future pending cb=[Task.task_wakeup()]> cb=[_chain_future.<locals>._call_set_state() at /usr/lib/python3.12/asyncio/futures.py:396]>
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-3264' coro=<PokeEnvClient.start_new_battle.<locals>._run_battle() running at /content/Smogon-RL/src/smogon_rl/pokeenv_client.py:204> wait_for=<Future pending cb=[_chain_future.<locals>._call_check_cancel() at /usr/lib/pytho

[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Battle update received: turn=22, finished=False.
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Waiting for turn advance: current_turn=22, previous_turn=22, elapsed=5.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=22, previous_turn=22, elapsed=10.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=22, previous_turn=22, elapsed=15.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=22, previous_turn=22, elapsed=20.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=22, previous_turn=22, elapsed=25.0s.
[PokeEnvClient] Turn-advance wait timed out at turn=22; returning last state.
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Battle update received: turn=23, finished=False.
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Battle update received: turn=24, finished=False.
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnv

In [19]:
# Cell 10: inspect GRPO sampling, then train on real rollout data
GRPO_ARGS = GRPOConfig(
    output_dir=f"debug_plain_trl_{NEXT_ADAPTER_REVISION}",
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=0.95,
    top_k=20,
    repetition_penalty=1.0,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    optim=OPTIM,
    max_steps=MAX_STEPS,
    num_train_epochs=1,
    loss_type="grpo",
    epsilon=0.2,
    beta=0.04,
    scale_rewards="group",
    max_prompt_length=512,
    logging_steps=1,
    save_steps=1000,
    report_to="none",
)

print("input_embeddings dtype:", model.get_input_embeddings().weight.dtype)
lm_head = getattr(model, "lm_head", None)
if lm_head is not None and hasattr(lm_head, "weight"):
    print("lm_head dtype:", lm_head.weight.dtype)

print("dataset len:", len(dataset))
print("sample row:", dataset[0])

probe_row = dataset[0]
test_completions = [
    [{"content": probe_row["collected_action"]}],
    [{"content": '{"action": "move", "choice": "thunderbolt"}'}],
    [{"content": "hello"}],
]
fr = format_reward([probe_row["prompt"]] * 3, test_completions)
gr = game_reward(
    [probe_row["prompt"]] * 3,
    test_completions,
    collected_action=[probe_row["collected_action"]] * 3,
    collected_reward=[probe_row["collected_reward"]] * 3,
)
print("format_reward:", fr)
print("game_reward:", gr)
print("total_reward:", [a + b for a, b in zip(fr, gr)])
assert (fr[0] + gr[0]) > (fr[1] + gr[1]) > (fr[2] + gr[2]), "GRPO reward ranking is broken"

# Inspect actual sampled completions on a real prompt before training.
messages = probe_row["prompt"]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    chat_template_kwargs={"enable_thinking": False},
)
device = model.get_input_embeddings().weight.device
inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

outs = model.generate(
    **inputs,
    max_new_tokens=MAX_NEW_TOKENS,
    num_return_sequences=8,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    top_k=20,
    repetition_penalty=1.0,
    pad_token_id=tokenizer.eos_token_id,
)

inp_len = inputs["input_ids"].shape[1]
decoded = [tokenizer.decode(o[inp_len:], skip_special_tokens=True) for o in outs]

print("\nGenerated completions for probe row:")
for i, d in enumerate(decoded):
    print(i, repr(d))

probe_completions = [[{"content": d}] for d in decoded]
probe_fr = format_reward([probe_row["prompt"]] * len(probe_completions), probe_completions)
probe_gr = game_reward(
    [probe_row["prompt"]] * len(probe_completions),
    probe_completions,
    collected_action=[probe_row["collected_action"]] * len(probe_completions),
    collected_reward=[probe_row["collected_reward"]] * len(probe_completions),
)
probe_total = [a + b for a, b in zip(probe_fr, probe_gr)]

print("\nProbe rewards:")
for i, (a, b, t) in enumerate(zip(probe_fr, probe_gr, probe_total)):
    print(i, "format=", a, "game=", b, "total=", t)

model.train()
if hasattr(model, "config"):
    model.config.use_cache = False

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, game_reward],
    args=GRPO_ARGS,
    train_dataset=dataset,
)

trainer.train()
print("GRPO completed successfully on real rollout data")


input_embeddings dtype: torch.float16
lm_head dtype: torch.float16
dataset len: 589
sample row: {'prompt': [{'content': 'You are a competitive Pokemon battler. Analyse the battle state below and choose exactly one action. You MUST output ONLY a single JSON object — no explanation, no markdown fences, just the raw JSON.\n{"action": "move" | "switch", "choice": "Exact Name of Move or Pokemon"}', 'role': 'system'}, {'content': '## Part A: Active Field\n### Active Self\n- Name: hitmonchan\n- HP: 100.0%\n- Status: Healthy\n- Ability: ironfist\n- Item: leftovers\n- Stat Modifiers: None\n### Active Opponent\n- Name: dugtrio\n- HP: 100.0%\n- Status: Healthy\n- Speed Range: 220-372\n\n## Part B: Full Self Roster\n- hitmonchan HP:100% OK Item:leftovers\n  Moves: closecombat(F120) | rapidspin(N20) | icepunch(I75) | machpunch(F40)\n- garchomp HP:100% OK Item:leftovers\n  Moves: earthquake(G100) | substitute(N0) | dragonclaw(D80) | swordsdance(N0)\n- rapidash HP:100% OK Item:leftovers\n  Moves: fla

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Generated completions for probe row:
0 '{"action": "switch", "choice": "vigoroth"}'
1 '{"action": "move", "choice": "closecombat"}'
2 '{"action": "switch", "choice": "garchomp"}'
3 '{"action": "move", "choice": "closecombat"}'
4 '{"action": "move", "choice": "closecombat"}'
5 '{"action": "move", "choice": "closecombat"}'
6 '{"action": "move", "choice": "closecombat"}'
7 '{"action": "switch", "choice": "garchomp"}'

Probe rewards:
0 format= 0.1 game= -0.25 total= -0.15
1 format= 0.1 game= 3.0 total= 3.1
2 format= 0.1 game= -0.25 total= -0.15
3 format= 0.1 game= 3.0 total= 3.1
4 format= 0.1 game= 3.0 total= 3.1
5 format= 0.1 game= 3.0 total= 3.1
6 format= 0.1 game= 3.0 total= 3.1
7 format= 0.1 game= -0.25 total= -0.15


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
1,0.032100
2,0.257000
3,0.336100
4,0.117800
5,0.029600
6,0.040400
7,0.058400
8,0.339800
9,0.223400
10,0.059500


GRPO completed successfully on real rollout data


In [20]:
%pip install -q huggingface_hub


In [ ]:
# Cell 11: Save LoRA adapter to Hugging Face Hub
import os
from huggingface_hub import login, create_repo

HF_REPO_ID = BASE_ADAPTER_REPO
HF_REVISION = NEXT_ADAPTER_REVISION

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError("HF_TOKEN is empty.")

login(token=hf_token)
create_repo(HF_REPO_ID, exist_ok=True, private=False)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved locally to {ADAPTER_DIR}")

model.push_to_hub(HF_REPO_ID, revision=HF_REVISION)
tokenizer.push_to_hub(HF_REPO_ID, revision=HF_REVISION)

print(f"Pushed to https://huggingface.co/{HF_REPO_ID} (revision: {HF_REVISION})")


Saved locally to ./openenv_grpo_lora_run3


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 60.9kB /  132MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpwlo9m6jf/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Pushed to https://huggingface.co/Atharva2099/openenv-smogon-rl (revision: grpo-qwen3-4b-run3)


2026-03-08 18:27:10,349 - RandomPlayer 33 - ERROR - no close frame received or sent
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/protocol.py", line 963, in transfer_data
    message = await self.read_message()
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/protocol.py", line 1033, in read_message
    frame = await self.read_data_frame(max_size=self.max_size)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/protocol.py", line 1108, in read_data_frame
    frame = await self.read_frame(max_size)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/protocol.py", line 1165, in read_frame
    frame = await Frame.read(
            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/websockets/legacy/framing.py", line 68, in read
  